<a href="https://colab.research.google.com/github/Emrul-Hasan-Emon/Generative-AI/blob/main/Data%20Preparation%20%26%20Cleaning/file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Data link: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

## Data Collection

In [9]:
import shutil
import kagglehub
import pandas as pd

In [5]:
# 1. Download the dataset via kagglehub
# This downloads and extracts the files into a cache directory
extracted_path = kagglehub.dataset_download(
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)
print("Downloaded and extracted to:", extracted_path)

# 2. Define destination path in the Colab content folder
# We specify 'imdb_reviews' as the base name for the zip file
destination_zip_base = "/content/imdb_reviews"

# 3. Zip the extracted folder and save it to /content/imdb_reviews.zip
zip_file_path = shutil.make_archive(
    base_name=destination_zip_base, format="zip", root_dir=extracted_path
)

print(f"Success! Your zip file is saved at: {zip_file_path}")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Downloaded and extracted to: /kaggle/input/imdb-dataset-of-50k-movie-reviews
Success! Your zip file is saved at: /content/imdb_reviews.zip


In [6]:
!unzip /content/imdb_reviews.zip

Archive:  /content/imdb_reviews.zip
  inflating: IMDB Dataset.csv        


In [7]:
data_path = "/content/IMDB Dataset.csv"

In [10]:
df = pd.read_csv(data_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
df.shape

(50000, 2)

There are many movie reviews present in the data set but we will work with only 100 otherwise it will take too much time to work with the data.

In [12]:
df = df.head(100)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Data Pre-processing

**Import Necessary Packages**

In [27]:
from bs4 import BeautifulSoup
import re

### Lower Case

In [13]:
df["review"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [14]:
df['review'] = df['review'].str.lower()

/tmp/ipykernel_16583/724319867.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].str.lower()


In [15]:
df["review"][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show is due to the fa

### HTML Tag Remove

**Option 1**: Using python BeautifulSoup Library

In [18]:
from bs4 import BeautifulSoup
import re

def remove_html_tags(text):
    # markup="html.parser" prevents warnings for short text snippets
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

**Option 2**: Manual Implementation

In [ ]:
import re
import html

def remove_html_tags_regex_advanced(text):
    if not isinstance(text, str):
        return text

    # 1. Remove script and style sections entirely (including content)
    clean_text = re.sub(r'<script\b[^<]*(?:(?!<\/script>)<[^<]*)*<\/script>', '', text, flags=re.IGNORECASE)
    clean_text = re.sub(r'<style\b[^<]*(?:(?!<\/style>)<[^<]*)*<\/style>', '', clean_text, flags=re.IGNORECASE)

    # 2. core tag removal, but ensuring it matches valid tag structures
    clean_text = re.sub(r'<[^>]+>', '', clean_text)

    # 3. Decode HTML entities (e.g., &nbsp; -> space, &amp; -> &)
    clean_text = html.unescape(clean_text)

    # 4. Clean up any leftover duplicate whitespaces
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

In [19]:
text = "<html><body><p> Movie 1</p><p> Actor - Aamir Khan</p><p> Click here to <a href='http://google.com'>download</a></p></body></html>"
remove_html_tags(text)

' Movie 1 Actor - Aamir Khan Click here to download'

In [20]:
df['review'] = df['review'].apply(remove_html_tags)

In [21]:
df["review"][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows wo

### Remove URL

In [22]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

In [23]:
text1 = 'Check out my youtube https://www.youtube.com/dswithbappy dswithbappy'
text2 = 'Check out my linkedin https://www.linkedin.com/in/boktiarahmed73/'
text3 = 'Google search here www.google.com'
text4 = 'For data click https://www.kaggle.com/'

In [28]:
remove_url(text1)

'Check out my youtube  dswithbappy'